# 01 · Download the thesis datasets into Google Drive

Run this in Google Colab while signed in as **san60lamichhane@gmail.com** (the Drive account).
Each dataset is pulled straight from its official source into Drive, so nothing passes through your laptop.

Drive layout created by this notebook:

```
MyDrive/PhD-AI-Cyber/
├── code/phd-ai-threat-detection/   ← this GitHub repo (code + results, also pushed to GitHub)
├── config/gated_urls.yaml          ← links you get after registering (kept out of GitHub)
└── data/raw/<dataset>/             ← raw data, Drive only, with MANIFEST.json (SHA-256 per file)
```

Safe to re-run at any time: finished files are skipped and interrupted downloads resume.
Colab sessions stop after a few hours, so for the big datasets just run the download cell again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
MYDRIVE    = '/content/drive/MyDrive'
DRIVE_ROOT = f'{MYDRIVE}/PhD-AI-Cyber'
REPO_NAME  = 'phd-ai-threat-detection'
REPO_DIR   = f'{DRIVE_ROOT}/code/{REPO_NAME}'
for d in ('code', 'config', 'data/raw'):
    pathlib.Path(DRIVE_ROOT, d).mkdir(parents=True, exist_ok=True)
print('Drive ready at', DRIVE_ROOT)

## Get the code

The repo is private, so Colab needs a GitHub token once. Create a fine-grained token at
https://github.com/settings/personal-access-tokens (signed in as `sandesh20lamichhane`, access to this one repo, *Contents: read and write*),
then add it in Colab's **🔑 Secrets** panel as `GITHUB_TOKEN` and switch on notebook access.

In [ ]:
from google.colab import userdata
try:
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
REMOTE = f'https://{TOKEN}@github.com/sandesh20lamichhane/{REPO_NAME}.git'
if pathlib.Path(REPO_DIR, '.git').exists():
    !git -C "$REPO_DIR" pull --ff-only "$REMOTE" main
elif pathlib.Path(REPO_DIR, 'src').exists():
    print('Using the copy of the code already in Drive (not a git clone yet).')
    print('Once GitHub is linked: rename that folder, re-run this cell, and the manifests are re-exported by --verify.')
else:
    !git clone "$REMOTE" "$REPO_DIR"
    # keep the token out of the copy saved in Drive
    !git -C "$REPO_DIR" remote set-url origin https://github.com/sandesh20lamichhane/$REPO_NAME.git
if pathlib.Path(REPO_DIR, '.git').exists():
    !git -C "$REPO_DIR" config user.name "Sandesh Lamichhane"
    !git -C "$REPO_DIR" config user.email "san30lamichhane@gmail.com"
%cd $REPO_DIR
!pip -q install -r requirements.txt


## Links that need registration (fill in once)

Three providers put their files behind a short form. Fill each form in your browser, copy the link you land on, and paste it below.
Leave a value empty until you have it; those datasets are simply skipped for now.

| Key | Where to register | What to paste |
|---|---|---|
| `ciciot2023_csv_folder` | https://www.unb.ca/cic/datasets/iotdataset-2023.html | the URL of the **CSV** folder listing (not PCAP) |
| `ciciomt2024_folder` | https://www.unb.ca/cic/datasets/iomt-dataset-2024.html | the URL of the **WiFi_and_MQTT/attacks/** folder listing |
| `lanl_cyber1_urls` | https://csr.lanl.gov/data/cyber1/ | the 5 file links: auth, proc, flows, dns, redteam (.txt.gz) |

UNSW-NB15 and ToN_IoT live in UNSW SharePoint folders that can't be scripted. Download the CSV folders listed in
`config/datasets.yaml` and drag them into `MyDrive/PhD-AI-Cyber/data/raw/unsw_nb15/` and `.../ton_iot/`; the next run checksums them.

In [ ]:
import yaml
GATED = {
    'ciciot2023_csv_folder': '',
    'ciciomt2024_folder': '',
    'lanl_cyber1_urls': [
        # 'https://.../auth.txt.gz',
    ],
}
path = pathlib.Path(DRIVE_ROOT, 'config', 'gated_urls.yaml')
saved = yaml.safe_load(path.read_text()) if path.exists() else {}
saved = saved or {}
saved.update({k: v for k, v in GATED.items() if v})   # never blank out a link saved earlier
path.write_text(yaml.safe_dump(saved))
print('Saved links for:', sorted(saved) or 'none yet')

## Optional: Kaggle fallback

Only used for UNSW-NB15 if you'd rather not do the SharePoint step. Add `KAGGLE_USERNAME` and `KAGGLE_KEY` to Colab secrets.
Official files are preferred for the thesis; the manifest records which source was used.

In [ ]:
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    !pip -q install kaggle
    print('Kaggle fallback enabled')
except Exception:
    print('No Kaggle secrets: fallback off (that is fine)')

## 1. Plan: list every file and its size, download nothing

In [ ]:
!python -m src.acquire.fetch --plan

## 2. Download

Downloads every enabled dataset that has what it needs. Stops a dataset before it would push raw data past 190 GB.
To do one dataset only: `!python -m src.acquire.fetch --only nsl_kdd`

In [ ]:
!python -m src.acquire.fetch

## 3. Verify checksums of everything in Drive

In [ ]:
!python -m src.acquire.fetch --verify

## 4. Save the manifests (results) to GitHub

In [ ]:
if pathlib.Path(REPO_DIR, '.git').exists():
    !git add results/data_manifest
    !git commit -m "Update dataset manifests" || echo "nothing new to commit"
    !git push "$REMOTE" HEAD:main
else:
    print('Not a git clone yet: manifests are saved in Drive under results/data_manifest/ and data/raw/*/MANIFEST.json')
